# 03 · KV Cache 与显存容量计算

这是**推理 infra 面试的必考题**，而且是最容易当场露馅的一题：

> 一台 80G 的卡，跑 Llama-3-8B，4K 上下文，最多能同时服务多少个请求？

答不上来的人，说明从没真正做过容量规划。这一章把公式推清楚、用代码验证、然后做成一个随时能用的计算器。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、先建立代价直觉：KV cache 省了多少计算

不用 KV cache 时，每生成一个 token 都要把前面所有 token 重新算一遍。prompt 越长，浪费越大——而且不是线性浪费，是**平方级**的。

In [ ]:
prompt_lens = [64, 256, 512]
N_OUT = 32

print(f"生成 {N_OUT} 个 token，对比两种实现的耗时\n")
print(f"{'prompt 长度':>12}{'无cache(ms)':>14}{'有cache(ms)':>14}{'加速比':>10}")
print("-" * 52)

for L in prompt_lens:
    p = torch.randint(0, model.cfg.vocab_size, (1, L), device=DEVICE)
    t_naive = bench(lambda: generate_naive(model, p, N_OUT), warmup=1, iters=3)
    t_cached = bench(lambda: generate_cached(model, p, N_OUT), warmup=1, iters=3)
    print(f"{L:>12}{t_naive:>14.1f}{t_cached:>14.1f}{t_naive / t_cached:>9.2f}x")

print()
print("prompt 越长，KV cache 的收益越大——因为没有它，每一步都要重算整段 prompt。")
print("但缓存不是免费的，代价就是显存。下面算它到底占多少。")

## 二、公式推导

每一层、每个 token、每个 KV head 都要存一份 K 和一份 V：

```
KV cache 字节数 = 2 × n_layer × n_kv_head × head_dim × seq_len × batch × dtype_bytes
                   ↑   ↑          ↑              ↑
                   |   |          |              └─ 每元素字节数（fp16 = 2）
                   |   |          └─ 每个头的维度
                   |   └─ KV head 的个数（GQA 下比 attention head 少）
                   └─ K 和 V 各一份
```

**两个高频陷阱**：

1. 忘记乘 2（K 和 V）。
2. 用 attention head 数计算。用了 GQA 的模型，KV head 数远小于 attention head 数，这是 GQA 存在的全部意义。

In [ ]:
# 在 MiniGPT 上实测，验证公式
idx = torch.randint(0, model.cfg.vocab_size, (2, 128), device=DEVICE)
_, past = model(idx)

actual = sum(t.numel() * t.element_size() for layer in past for t in layer)
theory = kv_bytes(model.cfg.n_layer, model.cfg.n_head, model.cfg.head_dim, seq_len=128, batch=2,
                  dtype_bytes=2)

print(f"实测 past 张量总字节: {actual / 1024:8.1f} KB")
print(f"公式计算值          : {theory / 1024:8.1f} KB")
print(f"一致                : {actual == theory}")
print()
print("公式是对的。注意 past 里每层有两个张量 (k, v)，shape 都是 (batch, n_head, seq, head_dim)。")

In [ ]:
# 每秒、每 token 的 KV cache —— 这是做容量规划时最常用的单位
def kv_per_token_kb(n_layer, n_kv_head, head_dim, dtype_bytes=2):
    return kv_bytes(n_layer, n_kv_head, head_dim, seq_len=1, dtype_bytes=dtype_bytes) / 1024


print(f"{'模型':<26}{'层数':>5}{'KV头':>6}{'每token':>12}{'4K上下文/请求':>16}")
print("-" * 68)
for name, cfg in [
    ("Llama-3-8B (GQA)", dict(n_layer=32, n_kv_head=8, head_dim=128)),
    ("Llama-3-8B (假如是 MHA)", dict(n_layer=32, n_kv_head=32, head_dim=128)),
    ("Qwen2.5-7B (GQA)", dict(n_layer=28, n_kv_head=4, head_dim=128)),
    ("Llama-3-70B (GQA)", dict(n_layer=80, n_kv_head=8, head_dim=128)),
]:
    per_tok = kv_per_token_kb(**cfg)
    four_k = per_tok * 4096 / 1024 / 1024  # KB → GiB
    print(f"{name:<26}{cfg['n_layer']:>5}{cfg['n_kv_head']:>6}{per_tok:>10.0f} KB{four_k:>14.2f} GB")

print()
print("两个值得记住的结论：")
print("  1. Llama-3-8B 每 token 约 128 KB，4K 上下文一个请求就要 0.5 GB。")
print("  2. 同样是 8B 模型，MHA 版本的 KV cache 是 GQA 的 4 倍——这就是为什么现在所有模型都用 GQA。")

## 三、容量计算器

现在把公式反过来用：给定卡和模型，算出最大并发。

In [ ]:
def capacity(gpu_mem_gb, params_b, n_layer, n_kv_head, head_dim, seq_len,
             weight_bytes=2, kv_elem_bytes=2, overhead_ratio=0.12):
    """返回 (最大并发, 总显存GB, 权重GB, 每请求KV GB)。

    overhead_ratio 预留激活值、CUDA graph、通信 buffer、显存碎片等开销。
    真实系统里 vLLM 还会把 block 内碎片算进去，这里取 12% 是个经验值。
    """
    total_bytes = gpu_mem_gb * (1 - overhead_ratio) * 1024 ** 3
    weight_bytes_total = params_b * 1e9 * weight_bytes
    avail = total_bytes - weight_bytes_total
    per_req = kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=kv_elem_bytes)
    if avail <= 0:
        return 0, gpu_mem_gb, weight_bytes_total / 1024 ** 3, per_req / 1024 ** 3
    return int(avail // per_req), gpu_mem_gb, weight_bytes_total / 1024 ** 3, per_req / 1024 ** 3


print("场景：80G 卡（如 H100/A100-80G），fp16 权重\n")
print(f"{'模型':<22}{'上下文':>8}{'权重GB':>9}{'每请求KB':>11}{'最大并发':>10}")
print("-" * 62)
for name, p, cfg in [
    ("Llama-3-8B", 8.03, dict(n_layer=32, n_kv_head=8, head_dim=128)),
    ("Qwen2.5-7B", 7.62, dict(n_layer=28, n_kv_head=4, head_dim=128)),
    ("Llama-3-70B", 70.6, dict(n_layer=80, n_kv_head=8, head_dim=128)),
]:
    for seq in [2048, 8192]:
        conc, _, wgb, perkb = capacity(80, p, seq_len=seq, **cfg)
        print(f"{name:<22}{seq:>8}{wgb:>9.1f}{perkb * 1024 * 1024:>11.0f}{conc:>10}")

### 把数字读一遍

几个可以直接用在面试里的结论：

- **8B 模型在 80G 卡上，2K 上下文约 220 并发，4K 掉到 110，8K 只剩 55**。这就是为什么线上要拆多副本，而不是指望单卡扛住全部流量。
- **70B 模型光权重就 130+ GB，单张 80G 卡连放都放不下**（上面表里会显示 0 并发）。必须先做张量并行切分或者量化到 FP8/INT4——这是"为什么需要模型并行"最直接的解释。
- **上下文长度翻倍，并发直接减半**。这解释了一个常见的线上现象：QPS 没变，只是把 `max_model_len` 调大了，服务却开始排队。

**想提高并发，按效果排序**：

| 手段 | 效果 | 代价 |
|---|---|---|
| 换 GQA/MLA 模型或降低 KV 精度 | KV 减半到 1/4 | 精度损失，需要评估 |
| 限制 `max_model_len` | 线性提升 | 长文档场景不可用 |
| 权重量化到 FP8/INT4 | 权重减半到 1/4，腾出空间给 KV | 精度损失 |
| 张量并行 TP | 每卡权重和 KV 都减半 | 通信开销，卡间互联要求高 |

## 四、面试怎么答这道题

被问到"80G 卡跑 8B 模型最多多少并发"，按这个顺序说：

1. **先问清楚前提**：上下文长度多少？KV 用 fp16 还是 fp8？权重什么精度？有没有其他模型混布？
2. **给出公式**：`2 × 层数 × KV头数 × head_dim × 序列长度 × 并发 × 字节数`，强调**用 KV head 而不是 attention head**（这是最常见的错误）。
3. **算权重**：8B × 2 字节 = 16 GB，再留 10% 左右的激活和碎片开销。
4. **算可用空间**：80G × 0.88 - 16G ≈ 54 GB。
5. **算每请求 KV**：8K 上下文约 0.5 GB → 大约 100 条并发。
6. **补一句工程现实**：这是理论值，实际还要扣掉 block 内碎片（第 05 章）、prefill 峰值显存、以及调度上要留的余量，通常按 70-80% 打折规划。

第 6 步是拉开差距的地方——说明你做过真实部署，不是只会套公式。

**作业**

1. 用 `capacity()` 算一下：如果 KV cache 量化到 fp8（`kv_elem_bytes=1`），并发能提升多少？
2. 如果要支撑 500 并发、8K 上下文跑 Llama-3-8B，需要几张 80G 卡？
3. 思考题：为什么 vLLM 的 PagedAttention 能把并发做得更高？（提示：和第 05 章的显存碎片有关）

**下一章**：既然单个请求喂不饱 GPU，怎么把不同长度、不同进度的请求拼进同一个 batch——continuous batching。